# S2 cointegration — FTMO 2-Step deep dive

After a firm/challenge is selected (FTMO 2-Step), inspect **which constraint binds**, drawdown budget at pass, min-days delay vs target-hit, funded survival, retries until first pass, and a leverage grid on **EV/day**.


## 0. Imports & Config


In [ ]:
import os
import sys

import pandas as pd
from IPython.display import display

cur = os.path.abspath(os.getcwd())
ROOT = cur
for _ in range(12):
    if os.path.isfile(os.path.join(cur, "pyproject.toml")) and os.path.isdir(
        os.path.join(cur, "06_risk")
    ):
        ROOT = cur
        break
    parent = os.path.dirname(cur)
    if parent == cur:
        break
    cur = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from risk.analytics.leverage.policy import k_fair_from_artifact
from risk.analytics.leverage.artifacts import load_leverage_artifact
from risk.analytics.monte_carlo.loaders import find_repo_root, load_sealed_s1, load_sealed_s2
from risk.analytics.prop_firm.registry import CHALLENGES, make_challenge
from risk.analytics.prop_firm.report import (
    binding_mix,
    fair_vs_ftmo_row,
    leverage_ev_grid,
    run_challenge_select,
    suggest_challenge_leverage,
)
from risk.analytics.prop_firm.s1_calendar import weekly_to_weekday_returns
from risk.analytics.prop_firm.plots import failure_mix_figure, leverage_heatmap_figure, retries_hist_figure

ROOT = find_repo_root(ROOT)
print("ROOT", ROOT)
print("registered challenges", sorted(CHALLENGES))


## 1. Load / selection


In [ ]:
RETURNS = load_sealed_s2(ROOT)
DEFAULT_H = 40
DEFAULT_HF = 40
ART = load_leverage_artifact(ROOT, "s2")
K_FAIR = k_fair_from_artifact(ART, RETURNS, periods_per_year=252.0)
print("daily bars", len(RETURNS), RETURNS.index.min().date(), RETURNS.index.max().date())
print("k_fair", K_FAIR)

N_SIM = 250
H = DEFAULT_H
HF = DEFAULT_HF
K = float(K_FAIR) if K_FAIR == K_FAIR else 1.0
CAP = 100000.0
FEE = 540.0
SPLIT = 0.80
print("n_returns", len(RETURNS), "default k", K)


## 2. Pathwise constraints


In [ ]:
pack = run_challenge_select(
    RETURNS,
    n_simulations=N_SIM,
    horizon=H,
    horizon_funded=HF,
    leverage=K,
    initial_capital=CAP,
    fee=FEE,
    profit_split=SPLIT,
    mean_block_length=10.0,
    random_seed=1,
)
res = pack["results"]
display(pack["headline"].to_frame("value"))
print("Challenge first binding among fails")
mix = binding_mix(res, phase="chal")
display(mix.to_frame("n"))
failure_mix_figure(mix, title="Challenge first binding").show()
print("Verification first binding among fails")
vmix = binding_mix(res, phase="verif")
display(vmix.to_frame("n"))
wins = res.loc[res["passed_challenge"]]
if len(wins):
    print("DD budget at challenge pass (equity - 90% floor)")
    display(wins["chal_dd_budget_at_pass"].describe().to_frame("dd_budget"))
    print("Min-days delay (passed_bar - target_hit_bar)")
    display(wins["chal_min_days_delay"].describe().to_frame("delay"))


## 3. Leverage grid

Maximize **EV/day**, not pass rate. `do_not_take` if EV or lower CI is not positive. Grid includes `p_fail_daily_loss` / `p_fail_max_loss`. Suggested $k$ is `min(k_fair, k_ftmo)` under those caps. Compare unconstrained fair VT vs FTMO-capped.


In [ ]:
grid = leverage_ev_grid(
    RETURNS,
    [0.5, 0.75, 1.0, 1.25, 1.5, 2.0],
    n_simulations=max(80, N_SIM // 2),
    horizon=H,
    horizon_funded=HF,
    initial_capital=CAP,
    fee=FEE,
    profit_split=SPLIT,
    mean_block_length=10.0,
    random_seed=2,
)
display(grid)
leverage_heatmap_figure(grid, value="ev_per_day").show()
leverage_heatmap_figure(grid, value="p_both").show()
sug = suggest_challenge_leverage(
    grid, k_fair=K_FAIR, max_p_fail_daily=0.40, max_p_fail_max=0.30
)
print("k_suggested = min(k_fair, k_ftmo)", sug)
display(pd.Series(sug, name="k_policy").to_frame("value"))
compare = fair_vs_ftmo_row(grid, sug)
print("unconstrained fair VT vs FTMO-capped")
display(compare)
best = grid.sort_values("ev_per_day", ascending=False).iloc[0]
print("best k by EV/day", float(best["leverage"]), "EV/day", float(best["ev_per_day"]), "do_not_take", bool(best["do_not_take"]))


## 4. Funded survival


In [ ]:
both = res.loc[res["passed_both"]]
if both.empty:
    print("No two-step passes in this storm — funded survival undefined.")
else:
    print("funded survive given both phases passed", float(both["funded_status"].eq("survived").mean()))
    display(both["funded_surplus"].describe().to_frame("surplus"))
retries_hist_figure(pack["retries_until_pass"]).show()


## 5. Evaluation

S2 sealed series is already **daily**. EOD book returns still cannot see true CEST intra-day equity or the calendar day a pair was opened; `|r|>eps` is the proxy.


In [ ]:
print("incomplete is not an FTMO fail; chal_incomplete rate is in the headline.")
print("EOD equity proxy: no intra-day high/low, no open PnL, no swaps.")
print("Trading-day proxy: |r| > 1e-12 (open date only cannot be seen on EOD book returns).")
